# Agricultural Data Cleaning and Processing
This notebook cleans the agricultural miscellaneous data from CSV file

In [8]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [9]:
# Load the data
file_path = '../../raw_data/agri_misc_data_v1.csv'
df = pd.read_csv(file_path)

# Display basic information
print("Original Data Shape:", df.shape)
print("\nFirst few rows:")
df.head()

Original Data Shape: (13, 9)

First few rows:


,Crop,2004-05,2005-06,2006-07,2007-08,2008-09,2009-10,2010-11,2011-12
0,Rice,100.0,101.0,99.0,105.0,112.0,121.0,117.0,110.0
1,Wheat,100.0,101.0,112.0,115.0,117.0,127.0,120.0,108.0
2,Coarse Cereals,100.0,107.0,110.0,115.0,113.0,123.0,122.0,136.0
3,Pulses,100.0,108.0,134.0,124.0,124.0,146.0,137.0,129.0
4,Vegetables,100.0,109.0,103.0,118.0,113.0,124.0,128.0,115.0


In [10]:
# Remove empty rows at the end
df = df.dropna(how='all')
print("After removing empty rows:", df.shape)

After removing empty rows: (12, 9)


In [11]:
# Rename columns for better readability
df.columns = ['Crop'] + [f'Year_{col}' for col in df.columns[1:]]

# Display cleaned column names
print("Cleaned Column Names:")
print(df.columns.tolist())

Cleaned Column Names:
['Crop', 'Year_2004-05', 'Year_2005-06', 'Year_2006-07', 'Year_2007-08', 'Year_2008-09', 'Year_2009-10', 'Year_2010-11', 'Year_2011-12']


In [12]:
# Check for missing values
print("Missing Values:\n")
print(df.isnull().sum())

# Check data types
print("\nData Types:")
print(df.dtypes)

Missing Values:

Crop            0
Year_2004-05    0
Year_2005-06    0
Year_2006-07    0
Year_2007-08    0
Year_2008-09    0
Year_2009-10    0
Year_2010-11    0
Year_2011-12    0
dtype: int64

Data Types:
Crop                str
Year_2004-05    float64
Year_2005-06    float64
Year_2006-07    float64
Year_2007-08    float64
Year_2008-09    float64
Year_2009-10    float64
Year_2010-11    float64
Year_2011-12    float64
dtype: object


In [13]:
# Convert year columns to numeric (if any are strings)
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Check for any conversion issues
print("After numeric conversion - Data Types:")
print(df.dtypes)

After numeric conversion - Data Types:
Crop                str
Year_2004-05    float64
Year_2005-06    float64
Year_2006-07    float64
Year_2007-08    float64
Year_2008-09    float64
Year_2009-10    float64
Year_2010-11    float64
Year_2011-12    float64
dtype: object


In [16]:
print("Missing values after conversion:")
print(df.isnull().sum())

# Fill only Crop column using forward fill
df['Crop'] = df['Crop'].ffill()

# Fill numeric columns with mean
for col in df.columns[1:]:
    df[col] = df[col].fillna(df[col].mean())

Missing values after conversion:
Crop            0
Year_2004-05    0
Year_2005-06    0
Year_2006-07    0
Year_2007-08    0
Year_2008-09    0
Year_2009-10    0
Year_2010-11    0
Year_2011-12    0
dtype: int64


In [17]:
# Remove any duplicate rows
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

After removing duplicates: (12, 9)


In [18]:
# Strip any whitespace from crop names
df['Crop'] = df['Crop'].str.strip()

# Display cleaned crop names
print("Unique crops in dataset:")
print(df['Crop'].unique())

Unique crops in dataset:
<StringArray>
[               'Rice',               'Wheat',      'Coarse Cereals',
              'Pulses',          'Vegetables',              'Fruits',
                'Milk', 'Eggs, Fish and Meat',            'Oilseeds',
           'Sugarcane',              'Fibers',     'All Agriculture']
Length: 12, dtype: str


In [19]:
# Reshape data from wide to long format for better analysis
df_long = pd.melt(df, 
                   id_vars=['Crop'], 
                   var_name='Year', 
                   value_name='Index_Value')

# Extract year from the column name
df_long['Year'] = df_long['Year'].str.replace('Year_', '')

# Sort by Crop and Year
df_long = df_long.sort_values(['Crop', 'Year']).reset_index(drop=True)

print("Long format data shape:", df_long.shape)
df_long.head()

Long format data shape: (96, 3)


,Crop,Year,Index_Value
0,All Agriculture,2004-05,100.0
1,All Agriculture,2005-06,99.0
2,All Agriculture,2006-07,101.0
3,All Agriculture,2007-08,104.0
4,All Agriculture,2008-09,106.0


In [20]:
# Calculate additional metrics
# 1. Year-over-year percentage change for each crop
df_long['YoY_Pct_Change'] = df_long.groupby('Crop')['Index_Value'].pct_change() * 100

# 2. Cumulative growth since base year (2004-05)
df_long['Cumulative_Growth'] = (df_long.groupby('Crop')['Index_Value'].transform(lambda x: x / x.iloc[0]) * 100)

print("Data with additional metrics:")
df_long.head(10)

Data with additional metrics:


,Crop,Year,Index_Value,YoY_Pct_Change,Cumulative_Growth
0,All Agriculture,2004-05,100.0,NaN,100.0
1,All Agriculture,2005-06,99.0,-1.000000,99.0
2,All Agriculture,2006-07,101.0,2.020202,101.0
3,All Agriculture,2007-08,104.0,2.970297,104.0
4,All Agriculture,2008-09,106.0,1.923077,106.0
5,All Agriculture,2009-10,115.0,8.490566,115.0
6,All Agriculture,2010-11,123.0,6.956522,123.0
7,All Agriculture,2011-12,122.0,-0.813008,122.0
8,Coarse Cereals,2004-05,100.0,NaN,100.0
9,Coarse Cereals,2005-06,107.0,7.000000,107.0


In [21]:
# Summary statistics
print("="*50)
print("SUMMARY STATISTICS")
print("="*50)
print("\nOverall Statistics:")
print(df_long['Index_Value'].describe())

print("\nStatistics by Crop:")
print(df_long.groupby('Crop')['Index_Value'].agg(['mean', 'min', 'max', 'std']))

print("\nYears in dataset:")
print(df_long['Year'].unique())

SUMMARY STATISTICS

Overall Statistics:
count     96.000000
mean     108.812500
std       13.739158
min       80.000000
25%      100.000000
50%      105.500000
75%      117.250000
max      146.000000
Name: Index_Value, dtype: float64

Statistics by Crop:
                        mean    min    max        std
Crop                                                 
All Agriculture      108.750   99.0  123.0   9.852483
Coarse Cereals       115.750  100.0  136.0  11.132321
Eggs, Fish and Meat  111.000   99.0  137.0  15.820421
Fibers               109.125   91.0  140.0  19.526081
Fruits               104.375   98.0  119.0   7.836499
Milk                 106.250   97.0  124.0  11.695542
Oilseeds              97.000   85.0  104.0   7.445037
Pulses               125.250  100.0  146.0  15.106763
Rice                 108.125   99.0  121.0   8.219098
Sugarcane             93.875   80.0  109.0  11.063938
Vegetables           113.750  100.0  128.0   9.676924
Wheat                112.500  100.0  127.0 

In [22]:
# Check for outliers using IQR method
Q1 = df_long['Index_Value'].quantile(0.25)
Q3 = df_long['Index_Value'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_long[(df_long['Index_Value'] < lower_bound) | (df_long['Index_Value'] > upper_bound)]
print(f"Number of potential outliers: {len(outliers)}")
if len(outliers) > 0:
    print("\nOutlier rows:")
    print(outliers)

Number of potential outliers: 1

Outlier rows:
      Crop     Year  Index_Value  YoY_Pct_Change  Cumulative_Growth
61  Pulses  2009-10        146.0       17.741935              146.0


In [23]:
# Create final cleaned datasets

# 1. Wide format (original structure but cleaned)
cleaned_wide = df.copy()

# 2. Long format (for time series analysis)
cleaned_long = df_long.copy()

# 3. Summary by crop (aggregated)
crop_summary = df_long.groupby('Crop').agg({
    'Index_Value': ['mean', 'min', 'max', 'std'],
    'YoY_Pct_Change': ['mean', 'std']
}).round(2)
crop_summary.columns = ['Mean_Index', 'Min_Index', 'Max_Index', 'Std_Index', 
                        'Mean_YoY_Change', 'Std_YoY_Change']
crop_summary = crop_summary.reset_index()

print("Crop Summary Table:")
crop_summary

Crop Summary Table:


,Crop,Mean_Index,Min_Index,Max_Index,Std_Index,Mean_YoY_Change,Std_YoY_Change
0,All Agriculture,108.75,99.0,123.0,9.85,2.94,3.62
1,Coarse Cereals,115.75,100.0,136.0,11.13,4.59,4.90
2,"Eggs, Fish and Meat",111.00,99.0,137.0,15.82,4.84,7.77
3,Fibers,109.12,91.0,140.0,19.53,5.51,12.33
4,Fruits,104.38,98.0,119.0,7.84,2.58,3.82
5,Milk,106.25,97.0,124.0,11.70,3.28,6.27
6,Oilseeds,97.00,85.0,104.0,7.45,0.62,8.86
7,Pulses,125.25,100.0,146.0,15.11,4.34,12.61
8,Rice,108.12,99.0,121.0,8.22,1.50,5.50
9,Sugarcane,93.88,80.0,109.0,11.06,1.76,14.75


In [27]:
import os

# Go 2 folders back + create Processed_data
output_dir = os.path.join("..", "..", "Processed_data")
os.makedirs(output_dir, exist_ok=True)

# 1. Wide format
wide_path = os.path.join(output_dir, "cleaned_agri_data_wide.csv")
cleaned_wide.to_csv(wide_path, index=False)
print(f"✓ Saved: {wide_path}")

# 2. Long format
long_path = os.path.join(output_dir, "cleaned_agri_data_long.csv")
cleaned_long.to_csv(long_path, index=False)
print(f"✓ Saved: {long_path}")

# 3. Crop summary
summary_path = os.path.join(output_dir, "crop_summary_statistics.csv")
crop_summary.to_csv(summary_path, index=False)
print(f"✓ Saved: {summary_path}")

# 4. Excel file
excel_path = os.path.join(output_dir, "cleaned_agri_data_complete.xlsx")
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    cleaned_wide.to_excel(writer, sheet_name='Wide_Format', index=False)
    cleaned_long.to_excel(writer, sheet_name='Long_Format', index=False)
    crop_summary.to_excel(writer, sheet_name='Crop_Summary', index=False)
****
print(f"✓ Saved: {excel_path}")

✓ Saved: ..\..\Processed_data\cleaned_agri_data_wide.csv
✓ Saved: ..\..\Processed_data\cleaned_agri_data_long.csv
✓ Saved: ..\..\Processed_data\crop_summary_statistics.csv
✓ Saved: ..\..\Processed_data\cleaned_agri_data_complete.xlsx


In [25]:
# Final verification of cleaned data
print("\n" + "="*50)
print("FINAL CLEANING SUMMARY")
print("="*50)
print(f"\nOriginal data shape: (13, 9)")
print(f"Cleaned wide format shape: {cleaned_wide.shape}")
print(f"Cleaned long format shape: {cleaned_long.shape}")
print(f"\nNumber of crops: {cleaned_wide['Crop'].nunique()}")
print(f"Year range: {cleaned_long['Year'].min()} to {cleaned_long['Year'].max()}")
print(f"Total records: {len(cleaned_long)}")
print(f"\nMissing values in cleaned data: {cleaned_long.isnull().sum().sum()}")
print(f"Duplicate rows in cleaned data: {cleaned_long.duplicated().sum()}")

print("\nâœ… Data cleaning completed successfully!")
print("\nGenerated files:")
print("1. cleaned_agri_data_wide.csv - Original wide format (cleaned)")
print("2. cleaned_agri_data_long.csv - Long format for time series analysis")
print("3. crop_summary_statistics.csv - Statistical summary by crop")
print("4. cleaned_agri_data_complete.xlsx - Excel file with all sheets")


FINAL CLEANING SUMMARY

Original data shape: (13, 9)
Cleaned wide format shape: (12, 9)
Cleaned long format shape: (96, 5)

Number of crops: 12
Year range: 2004-05 to 2011-12
Total records: 96

Missing values in cleaned data: 12
Duplicate rows in cleaned data: 0

âœ… Data cleaning completed successfully!

Generated files:
1. cleaned_agri_data_wide.csv - Original wide format (cleaned)
2. cleaned_agri_data_long.csv - Long format for time series analysis
3. crop_summary_statistics.csv - Statistical summary by crop
4. cleaned_agri_data_complete.xlsx - Excel file with all sheets
